In [1]:
%cd /home/dramco_spark/6GTandem_RT_server/
# All the necessary imports
import os
import yaml
import sionna.rt
import time
import xarray as xr
import mitsuba as mi
import numpy as np
import matplotlib.pyplot as plt

from sionna.rt import (
    load_scene,
    PlanarArray,
    Transmitter,
    Receiver,
    PathSolver,
    Camera,
    subcarrier_frequencies,
)
from src.utils import (
    ituf_glass_callback,
    ituf_concrete_callback,
    ituf_metal_callback,
    ituf_polystyrene_callback,
    ituf_mdf_callback,
)
from src.patterns import MeasuredPattern
from sionna.rt import ITURadioMaterial

jitc_llvm_init(): LLVM API initialization failed ..


/home/dramco_spark/6GTandem_RT_server


In [2]:
# Select which environment you want to verify
environment = "industry_hall"

In [3]:
# For the custom materials, use an ITU material and change its callback.
def custom_mat(props, callback):
    itu_material = ITURadioMaterial(props=props)
    itu_material.frequency_update_callback = callback

    return itu_material

# Custom material BSDFs. These must match with the BSDF names in the .xml file.
# In the XML file the BSDF material must have a <string name="type" value="glass"/>
# where value is an existing ITU material.
mi.register_bsdf("custom_glass", lambda props: custom_mat(props, ituf_glass_callback))
mi.register_bsdf("custom_polystyrene", lambda props: custom_mat(props, ituf_polystyrene_callback))
mi.register_bsdf("custom_concrete", lambda props: custom_mat(props, ituf_concrete_callback))
mi.register_bsdf("custom_mdf", lambda props: custom_mat(props, ituf_mdf_callback))
mi.register_bsdf("custom_metal", lambda props: custom_mat(props, ituf_metal_callback))

def check_materials(config, scene):
    # check conductivity and relative permittivity at different frequencies
    # loop through material names and print them
    sub_GHz = config["sub10GHz_config"]["fc"]
    sub_THz = config["subTHz_config"]["fc"]
    print(f"Checking materials at {sub_GHz / 1e9} GHz and {sub_THz / 1e9} GHz")
    for key, value in scene.objects.items():
        print(f"---------------{key=}----------------")
        # Print name of assigned radio material for different frequenies
        for f in [sub_GHz, sub_THz]:  # Print for differrent frequencies
            scene.frequency = f
            value.radio_material.frequency_update()  # update the frequency of the objects
            print(f"\nRadioMaterial: {value.radio_material.name} at {scene.frequency[0] / 1e9} GHz")
            print(f"Conductivity: {value.radio_material.conductivity.numpy()}")
            print(f"Relative permittivity: {value.radio_material.relative_permittivity.numpy()}")
            print(f"Scattering coefficient: {value.radio_material.scattering_coefficient.numpy()}")
            print(f"XPD coefficient: {value.radio_material.xpd_coefficient.numpy()}")

In [ ]:
# Necessary setup code.
# Load config file.
with open(f"environments/{environment}/config.yaml", 'r') as file:
    config = yaml.safe_load(file)

# Register antenna patterns.
antenna_path = config["paths"]["antenna_rad_path"]

# Register one measured pattern.
path = os.path.join(antenna_path, f"element1.csv")

def measured_pattern_factory(csv_path=path, normalize=False, **kwargs):
    """Factory method that returns an instance of the antenna pattern"""
    return MeasuredPattern(csv_path=csv_path, normalize=normalize)

# Register it under a custom name.
sionna.rt.register_antenna_pattern("custom_measured_element", measured_pattern_factory)

# load scene
scene = load_scene(config["paths"]["scenepath"])

# Check that the right custom materials are set.
check_materials(config, scene)

# configure tx and rx arrays
antenna_conf = config["antenna_config"]
N_antennas = config["antenna_config"]["N_antennas"]

if antenna_conf["pattern"] == "measured":
    pattern = "custom_measured_element"
elif antenna_conf["pattern"] == "tr38901":
    pattern = "tr38901"
else:
    raise ValueError(f"Invalid antenna pattern selected: {antenna_conf['pattern']}")

scene.tx_array = PlanarArray(
    num_cols=N_antennas,
    num_rows=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization=config["antenna_config"]["polarization"],
)

# Configure antenna array for all receivers
scene.rx_array = PlanarArray(
    num_cols=N_antennas,
    num_rows=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization=config["antenna_config"]["polarization"],
)

# sub-THz stripe specs
stripe_start_pos = config["stripe_config"]["stripe_start_pos"]
stripe_end_pos = config["stripe_config"]["stripe_end_pos"]
N_RUs = config["stripe_config"]["N_RUs"]  # adjust to size of the room (along y axis)
N_stripes = config["stripe_config"]["N_stripes"]  # adjust to size of the room (alang x axis)
total_N_RUs = N_RUs * N_stripes  # total number of radio units
space_between_RUs = config["stripe_config"]["space_between_RUs"]  # in meters
space_between_stripes = config["stripe_config"]["space_between_stripes"]  # in meters
stripe_direction = config["stripe_config"]["stripe_direction"]

# Compute the RU positions.
x_pos = np.linspace(stripe_start_pos[0], stripe_end_pos[0], N_RUs).tolist()
y_pos = np.linspace(stripe_start_pos[1], stripe_end_pos[1], N_RUs).tolist()
z_pos = np.linspace(stripe_start_pos[2], stripe_end_pos[2], N_RUs).tolist()
# Perform a sanity check to see if the positions are ok.
x_diff = x_pos[1] - x_pos[0]
y_diff = y_pos[1] - y_pos[0]
z_diff = z_pos[1] - z_pos[0]
ru_dist = np.sqrt(x_diff ** 2 + y_diff ** 2 + z_diff ** 2) 

assert np.isclose(ru_dist, space_between_RUs), f"Actual space between RUs {ru_dist} does not match the one specified in the config {space_between_RUs}"

# OFDM system parameters
BW = config["subTHz_config"]["bw"]  # Bandwidth of the system
num_subcarriers = config["subTHz_config"]["num_subcarriers"]

subcarrier_spacing = BW / num_subcarriers
frequencies = subcarrier_frequencies(
    num_subcarriers, subcarrier_spacing
)  # Compute baseband frequencies of subcarriers relative to the carrier frequency

# set scene frequency
scene.frequency = config["subTHz_config"]["fc"]  # Set frequency to fc

# Instantiate a path solver
# The same path solver can be used with multiple scenes
p_solver = PathSolver()

Checking materials at 3.5 GHz and 157.75 GHz
---------------key='Plane'----------------

RadioMaterial: ituf_concrete at 3.5 GHz
Conductivity: [0.01019339]
Relative permittivity: [1.9]
Scattering coefficient: [0.]
XPD coefficient: [0.]

RadioMaterial: ituf_concrete at 157.750001664 GHz
Conductivity: [1.4400915]
Relative permittivity: [1.9]
Scattering coefficient: [0.]
XPD coefficient: [0.]
---------------key='no-name-1'----------------

RadioMaterial: ituf_metal at 3.5 GHz
Conductivity: [1.e+07]
Relative permittivity: [1.]
Scattering coefficient: [0.]
XPD coefficient: [0.]

RadioMaterial: ituf_metal at 157.750001664 GHz
Conductivity: [1.e+07]
Relative permittivity: [1.]
Scattering coefficient: [0.]
XPD coefficient: [0.]


In [5]:
# Generate user locations
# set seed
np.random.seed(2025)

def is_point_invalid(x, y, z):
    """Check that the point is not inside an object."""
    # Check if it is not inside any of the objects.
    for object in objects.values():
        if object[0] <= x <= object[1] and object[2] <= y <= object[3]:
            return True

    # Otherwise, point is outside all objects.
    return False


# grid under each RU 
stripe_start_pos = config['stripe_config']['stripe_start_pos'] 
array_direction = config["stripe_config"]["array_direction"]
N_RUs = config['stripe_config']['N_RUs']# adjust to size of the room (along y axis)
N_stripes = config['stripe_config']['N_stripes'] # adjust to size of the room (alang x axis)
space_between_RUs = config['stripe_config']['space_between_RUs'] # in meters
space_between_stripses = config['stripe_config']['space_between_stripes'] # in meters
ue_config = config["ue_locations_config"]
ue_points = ue_config['num_locations']
simulation_area = ue_config["ue_area"]
safety_offset = ue_config["safety_offset"]
z_height = ue_config["z_height"]
objects = config["objects"]

x_points = np.random.uniform(simulation_area[0] + safety_offset, simulation_area[1]  - safety_offset, ue_points)
y_points = np.random.uniform(simulation_area[2] + safety_offset, simulation_area[3] - safety_offset, ue_points)
samples = np.column_stack((x_points, y_points, z_height * np.ones(ue_points)))

nr_ue_locs = ue_points + (N_RUs * N_stripes)

# generate dataset of ue locations
samples_grid = np.zeros((N_RUs * N_stripes, 3))
stripe_labels = []
ru_labels = []
for stripe_idx in range(N_stripes):
    for RU_idx in range(N_RUs):
        # compute RU position
        rux = x_pos[RU_idx]
        ruy = y_pos[RU_idx]

        if stripe_direction == "y":
            rux += stripe_idx * space_between_stripes
        elif stripe_direction == "x":
            ruy += stripe_idx * space_between_stripes
        else:
            raise ValueError(f"Invalid stripe direction: {stripe_direction}")

        samples_grid[stripe_idx * N_RUs + RU_idx, :] = [rux, ruy, z_height]
        stripe_labels.append(stripe_idx)
        ru_labels.append(RU_idx)

# Combine samples into a single array
all_samples = np.vstack([samples, samples_grid])
nr_ue_locs = all_samples.shape[0]

invalid_point_labels = []
for s in all_samples:
    invalid_point = is_point_invalid(*s)
    invalid_point_labels.append(invalid_point)

zone_labels = np.array(['Zone 1'] * samples.shape[0] + ['Grid'] * samples_grid.shape[0])

# Additional boolean to check if the ue is under the stripe grid or in a zone.
ue_on_stripe_grid = np.array([False] * samples.shape[0] + [True] * samples_grid.shape[0])

stripe_labels = np.array([np.nan] * samples.shape[0] + stripe_labels)
ru_labels = np.array([np.nan] * samples.shape[0] + ru_labels)

# unique id per user
user_ids = np.arange(nr_ue_locs)

# Create the Dataset
ds_users = xr.Dataset(
    data_vars={
        "user_id": ("user", user_ids),
        "x": ("user", all_samples[:, 0].astype(np.float32)),
        "y": ("user", all_samples[:, 1].astype(np.float32)),
        "z": ("user", all_samples[:, 2].astype(np.float32)),
        "zone": ("user", zone_labels),
        "ue_stripe_idx": ("user", stripe_labels),
        "ue_ru_idx": ("user", ru_labels),
        "ue_on_stripe_grid": ("user", ue_on_stripe_grid),
        "invalid_point": ("user", invalid_point_labels)
    }
)

In [ ]:
"""A preview showing all the RUs and the UE locations."""

# By default the antenna array points towards the x-axis.
# The orientation specifies three angles, yaw, pitch and roll respectively.
# Yaw is a rotation around the z-axis, Pitch is around the y-axis and Roll is around the x-axis.
# So setting pitch to 90deg rotates the RU downwards while the yaw rotates the antenna array orientation.
# This means that only applying pitch makes the array stay aligned along the y-axis while applying yaw makes
# it aligned along the x-axis.
yaw = 0
if array_direction == "x":
    yaw = np.pi / 2

# Add all the radio units to the scene
for stripe_idx in range(N_stripes):
    for RU_idx in range(N_RUs):
        rux = x_pos[RU_idx]
        ruy = y_pos[RU_idx]
        ruz = z_pos[RU_idx]

        if stripe_direction == "y":
            rux += stripe_idx * space_between_stripes
        elif stripe_direction == "x":
            ruy += stripe_idx * space_between_stripes
        else:
            raise ValueError(f"Invalid stripe direction: {stripe_direction}")

        # Create RU transmitter instance
        orientation = mi.Point3f(yaw, np.pi/2, 0)
        pos = mi.Point3f(rux, ruy, ruz)
        tx = Transmitter(name=f"tx_stripe_{stripe_idx}_RU_{RU_idx}", position=pos, display_radius=0.1, orientation=orientation)

        # Add RU transmitter instance to scene
        scene.add(tx)

for ue_idx in range(ds_users.sizes["user"]):
    # UE points at an invalid location are skipped.
    if ds_users.invalid_point.values[ue_idx]:
        continue
    
    # get coordinates
    x, y, z = ds_users.x.values[ue_idx], ds_users.y.values[ue_idx], ds_users.z.values[ue_idx]
    ue_pos = mi.Point3f(float(x), float(y), float(z))

    # Create a receiver
    orientation = mi.Point3f(yaw, -np.pi/2, 0)
    rx = Receiver(name=f"rx_{ue_idx}", position=ue_pos, display_radius=0.1, orientation=orientation)

    # Add receiver instance to scene
    scene.add(rx)

# Create new camera with different configuration
my_cam = Camera(**config["render_config"])

# Either render the scene to a jpg file or show a preview.
scene.render_to_file(camera=my_cam, filename='scene_verification.png', resolution=(650, 500), num_samples=512, clip_at=20) # Increase num_samples to increase image quality
#scene.preview()

# Now clean up by removing all RUs and UEs again.
for stripe_idx in range(N_stripes):
    for RU_idx in range(N_RUs):
        scene.remove(f"tx_stripe_{stripe_idx}_RU_{RU_idx}")

for ue_idx in range(ds_users.sizes["user"]):
    scene.remove(f"rx_{ue_idx}")

In [7]:
"""Plot the rays for one UE and every RU."""

# UE number to use for ray tracing.
ue_idx = 0
# get coordinates
x, y, z = ds_users.x.values[ue_idx], ds_users.y.values[ue_idx], ds_users.z.values[ue_idx]
ue_pos = mi.Point3f(float(x), float(y), float(z))

# Create a receiver
orientation = mi.Point3f(yaw, -np.pi/2, 0)
rx = Receiver(name=f"rx_{ue_idx}", position=ue_pos, display_radius=0.5, orientation=orientation)

# Add receiver instance to scene
scene.add(rx)

# Preallocate channel tensor and index arrays (2x N^2 because cross polarization)
channel_tensor = np.empty((total_N_RUs, N_antennas, N_antennas, num_subcarriers), dtype=np.complex64)
stripe_idx_arr = np.empty(total_N_RUs, dtype=np.int32)
ru_idx_arr = np.empty(total_N_RUs, dtype=np.int32)
tx_idx = 0

# loop over all stripes
for stripe_idx in range(N_stripes):
    # loop over all RUs
    for RU_idx in range(N_RUs):
        rux = x_pos[RU_idx]
        ruy = y_pos[RU_idx]
        ruz = z_pos[RU_idx]

        if stripe_direction == "y":
            rux += stripe_idx * space_between_stripes
        elif stripe_direction == "x":
            ruy += stripe_idx * space_between_stripes
        else:
            raise ValueError(f"Invalid stripe direction: {stripe_direction}")

        # Create RU transmitter instance
        orientation = mi.Point3f(yaw, np.pi/2, 0)
        pos = mi.Point3f(rux, ruy, ruz)
        tx = Transmitter(name=f"tx_stripe_{stripe_idx}_RU_{RU_idx}", position=pos, display_radius=0.5, orientation=orientation)

        # Add RU transmitter instance to scene
        scene.add(tx)

        # todo recheck this
        paths = p_solver(
            scene=scene,
            max_depth=5,
            los=True,
            specular_reflection=True,
            diffuse_reflection=False,  # no scattering
            refraction=True,
            synthetic_array=False,
            seed=41,
        )

        scene.render_to_file(camera=my_cam, filename=f'scene_with_stripe_{stripe_idx}_RU_{RU_idx}.png', 
                resolution=(650, 500), num_samples=512, clip_at=20, paths=paths)
        #scene.preview(paths=paths, resolution=(1000, 600), clip_at=20)

        # Compute channel frequency response
        # Shape: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_time_steps, num_subcarriers]
        h_freq = paths.cfr(frequencies=frequencies, normalize_delays=True, out_type="numpy")

        h_freq = np.squeeze(h_freq)

        # plug into channel tensor
        channel_tensor[tx_idx] = h_freq

        # assign stripe and ru idx
        stripe_idx_arr[tx_idx] = stripe_idx
        ru_idx_arr[tx_idx] = RU_idx

        # increment tx idx counter
        tx_idx += 1

        # remove tx from the scene after computation
        scene.remove(f"tx_stripe_{stripe_idx}_RU_{RU_idx}")

# remove rx from the scene after computation
scene.remove(f"rx_{ue_idx}")

/home/dramco_spark/6GTandem_RT_server/.venv/lib/python3.11/site-packages/drjit/ast.py:838: RuntimeWarning: The AST-transforming decorator @drjit.syntax was called more than 1000 times by your program. Since transforming and recompiling Python code is a relatively expensive operation, it should not be used within loops or subroutines. Please move the function to be transformed to the top program level and decorate it there.
  warnings.warn(


TypeError: cannot unpack non-iterable NoneType object